# Estimation of Protein Corona Thickness by Differential Centrifugal Sedimentation (CPS/DCS)

**Model:** The reference AuNP is represented by a gold core and a hypothetical citrate layer. Serum-exposed AuNPs are represented by the same gold core and a protein shell, assuming replacement of citrate by the corona. This notebook applies **relative calibration to the bare AuNP CPS measurement**. The calibration is an adaptation of sedimentation principles discussed by Davidson et al. (2017), **not an exact reproduction of their protocol**.

**Interpretation:** The resulting thicknesses are model-dependent estimates, not direct thickness measurements. More than one mathematical root can correspond to the same apparent CPS diameter.

**Reference:** Davidson et al., *Analytical Chemistry* (2017), DOI: [10.1021/acs.analchem.7b01229](https://doi.org/10.1021/acs.analchem.7b01229).

## 1. Experimental measurements and model assumptions

The TEM diameter describes the **physical gold core**, whereas the CPS measurements are **apparent sedimentation diameters**. The citrate thickness, citrate density, corona density, and fluid density are modeling inputs and must be checked against experimental conditions. The assumed citrate shell is present only in the reference model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from pathlib import Path
import csv

# Experimental measurements (nm)
D_TEM = 18.98             # PHYSICAL gold-core diameter
D_CPS_AUNP = 14.48        # APPARENT bare-AuNP diameter
D_CPS = {"RS": 10.75, "RC": 10.34}

# Assumed model parameters (verify against experimental conditions)
RHO_AU = 19.3             # g/cm³
RHO_CITRATO = 1.40        # g/cm³ (assumed)
RHO_CORONA = 1.15         # g/cm³ (assumed hydrated corona)
RHO_FLUIDO = 1.064        # g/cm³ (verify CPS fluid/gradient)
T_CITRATO = 1.0          # nm, hypothetical radial thickness

OUTPUT_DIR = Path('cps_corona_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Core–shell density and relative CPS model

For the **reference particle** (gold core + citrate):

$$D_0=D_{Au}+2t_{cit}$$

$$\rho_{eff,0}=\frac{\rho_{Au}D_{Au}^{3}+\rho_{cit}(D_0^3-D_{Au}^3)}{D_0^3}$$

For the **serum-exposed particle** (gold core + protein corona, without a separate citrate layer):

$$D=D_{Au}+2t_{cor}$$

$$\rho_{eff}(t_{cor})=\frac{\rho_{Au}D_{Au}^{3}+\rho_{cor}(D^3-D_{Au}^3)}{D^3}$$

The notebook uses the following **relative calibration**:

$$d_{CPS,pred}=d_{CPS,AuNP}\sqrt{\frac{D^2[\rho_{eff}(t_{cor})-\rho_f]}{D_0^2(\rho_{eff,0}-\rho_f)}}$$

The unknown radial corona thickness is found by solving $d_{CPS,pred}=d_{CPS,measured}$. The approach assumes that the same relative calibration factor applies to reference and serum-exposed particles.

In [ ]:
REFERENCE_DIAMETER = D_TEM + 2 * T_CITRATO

def reference_effective_density():
    return (RHO_AU * D_TEM**3 + RHO_CITRATO * (REFERENCE_DIAMETER**3 - D_TEM**3)) / REFERENCE_DIAMETER**3

def total_particle_diameter(corona_thickness):
    return D_TEM + 2 * np.asarray(corona_thickness)

def corona_effective_density(corona_thickness):
    D = total_particle_diameter(corona_thickness)
    return (RHO_AU * D_TEM**3 + RHO_CORONA * (D**3 - D_TEM**3)) / D**3

def predicted_cps_diameter(corona_thickness):
    D = total_particle_diameter(corona_thickness)
    num = D**2 * (corona_effective_density(corona_thickness) - RHO_FLUIDO)
    den = REFERENCE_DIAMETER**2 * (reference_effective_density() - RHO_FLUIDO)
    if den <= 0:
        raise ValueError('The reference has no positive density contrast.')
    if np.any(num < 0):
        raise ValueError('Negative density contrast: review model parameters.')
    return D_CPS_AUNP * np.sqrt(num / den)

def find_corona_thickness_roots(measured_cps, t_max=200.0, n_points=10001):
    """Find roots in [0, t_max] using sign changes and Brent's method."""
    t = np.linspace(0, t_max, n_points)
    residual = predicted_cps_diameter(t) - measured_cps
    roots = []
    for i in range(len(t) - 1):
        if np.isclose(residual[i], 0, atol=1e-10):
            root = float(t[i])
        elif residual[i] * residual[i + 1] < 0:
            root = brentq(lambda x: float(predicted_cps_diameter(x) - measured_cps), float(t[i]), float(t[i+1]))
        else:
            continue
        if not any(abs(root - r) < 1e-5 for r in roots):
            roots.append(root)
    return roots

print(f'TEM gold-core diameter: {D_TEM:.2f} nm')
print(f'Assumed reference physical diameter with citrate: {REFERENCE_DIAMETER:.2f} nm')
print(f'Reference effective density: {reference_effective_density():.4f} g/cm³')
print(f'Predicted CPS diameter for zero corona thickness: {float(predicted_cps_diameter(0)):.2f} nm')

## 3. Numerical solution and export of all candidate thicknesses

A grid spanning 0–200 nm is used to detect sign changes in the difference between predicted and measured CPS diameters. Each bracketed root is refined using `scipy.optimize.brentq`. **All roots are retained**, because the model is not necessarily monotonic. Results include radial corona thickness, estimated physical diameter, effective density, and predicted CPS diameter.

In [ ]:
results = []
for group, measured_cps in D_CPS.items():
    roots = find_corona_thickness_roots(measured_cps)
    if not roots:
        print(f'WARNING: no roots found for {group}; review assumptions or increase t_max.')
    for solution_number, t in enumerate(roots, 1):
        results.append({
            'Group': group,
            'Solution': solution_number,
            'Radial corona thickness (nm)': float(t),
            'Total physical diameter (nm)': float(total_particle_diameter(t)),
            'Measured CPS diameter (nm)': measured_cps,
            'Predicted CPS diameter (nm)': float(predicted_cps_diameter(t)),
            'Effective density (g/cm3)': float(corona_effective_density(t)),
        })

print('\nALL DETECTED MATHEMATICAL SOLUTIONS')
for record in results:
    print(f"{record['Group']} | solution {record['Solution']}: "
          f"t = {record['Radial corona thickness (nm)']:.3f} nm; "
          f"D = {record['Total physical diameter (nm)']:.3f} nm; "
          f"Predicted CPS = {record['Predicted CPS diameter (nm)']:.3f} nm")

csv_file = OUTPUT_DIR / 'all_mathematical_solutions.csv'
with csv_file.open('w', newline='', encoding='utf-8-sig') as f:
    if results:
        writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
        writer.writeheader()
        writer.writerows(results)
print('CSV saved to:', csv_file.resolve())

## 4. Model response curve and candidate solutions

The continuous curve shows the predicted apparent CPS diameter as a function of **radial** protein-shell thickness. Horizontal lines indicate measured CPS diameters for healthy-serum (RS) and cancer/cachexia-serum (RC) coronas. Markers identify **all detected intersections**. Figures are exported as PNG and SVG.

In [ ]:
t = np.linspace(0, 200, 1500)
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(t, predicted_cps_diameter(t), color='#253b6e', linewidth=2.2, label='Model-predicted CPS diameter')
colors = {'RS': '#169c9c', 'RC': '#d65a54'}
for group, measured_cps in D_CPS.items():
    ax.axhline(measured_cps, color=colors[group], linestyle='--', alpha=0.8,
               label=f'{group}: measured CPS = {measured_cps:.2f} nm')
    for root in find_corona_thickness_roots(measured_cps):
        ax.scatter([root], [measured_cps], color=colors[group], s=65, zorder=5)
        ax.annotate(f'{root:.1f} nm', (root, measured_cps), xytext=(5, 8),
                    textcoords='offset points', fontsize=9)
ax.set(xlabel='Radial corona thickness (nm)', ylabel='Apparent CPS diameter (nm)',
       title='Gold-core/corona model: multiple possible solutions')
ax.legend(fontsize=9)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'cps_model_response_curve.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'cps_model_response_curve.svg', bbox_inches='tight')
plt.show()

## 5. Physical-diameter estimates and measured CPS diameters

The first plot decomposes the **estimated physical diameter** into the TEM gold-core diameter and the corona's **diametral contribution** ($2\times$ radial thickness). For visualization **only**, it uses the smallest positive root. The second plot shows the **measured apparent CPS diameters**. These are different physical quantities and must not be interpreted on the same basis. The smallest root has not been independently validated.

In [ ]:
# Comparison plots: estimated physical diameters versus measured apparent CPS diameters
# Corona thicknesses are model estimates, not direct measurements.

colors = {'AuNP': '#C4A04A', 'RS': '#2D9C91', 'RC': '#CB6476'}
groups = ['AuNP', 'RS', 'RC']
smallest_roots = {}
for group in ('RS', 'RC'):
    options = [l for l in results if l['Group'] == group]
    if not options:
        raise ValueError(f'No solution for {group}; review model assumptions.')
    smallest_roots[group] = min(options, key=lambda l: l['Radial corona thickness (nm)'])

core = np.array([D_TEM] * 3, dtype=float)
corona_diametral = np.array([
    0,
    2 * smallest_roots['RS']['Radial corona thickness (nm)'],
    2 * smallest_roots['RC']['Radial corona thickness (nm)'],
])

fig, ax = plt.subplots(figsize=(9, 6))
x = np.arange(len(groups))
ax.bar(x, core, width=0.58, color='#C4A04A', label='Gold core (TEM)')
ax.bar(x, corona_diametral, width=0.58, bottom=core, color=[colors[g] for g in groups],
       label='Corona contribution (2 × radial thickness)')
for i, (n, c) in enumerate(zip(core, corona_diametral)):
    ax.text(i, n / 2, f'{n:.2f} nm', ha='center', va='center', fontsize=10, fontweight='bold')
    if c > 0:
        ax.text(i, n + c / 2, f'{c:.2f} nm\n(t = {c/2:.2f} nm)',
                ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    ax.text(i, n + c + 1.0, f'Total: {n+c:.2f} nm', ha='center', fontsize=10)
ax.set_xticks(x, ['AuNP', 'AuNP + RS', 'AuNP + RC'])
ax.set_ylabel('Physical diameter (nm)')
ax.set_title('Estimated physical diameter: gold core and protein corona')
ax.set_ylim(0, max(core + corona_diametral) * 1.25)
ax.legend(loc='upper left')
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'estimated_physical_diameters.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'estimated_physical_diameters.svg', bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
cps_values = [D_CPS_AUNP, D_CPS['RS'], D_CPS['RC']]
bars = ax.bar(['AuNP', 'AuNP + RS', 'AuNP + RC'], cps_values,
                color=[colors[g] for g in groups], width=0.58)
for bar, value in zip(bars, cps_values):
    ax.text(bar.get_x() + bar.get_width()/2, value + 0.25,
            f'{value:.2f} nm', ha='center', fontsize=11)
ax.set_ylabel('Measured apparent CPS diameter (nm)')
ax.set_title('Experimental apparent CPS diameters')
ax.set_ylim(0, max(cps_values) * 1.2)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'measured_apparent_cps_diameters.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'measured_apparent_cps_diameters.svg', bbox_inches='tight')
plt.show()

print('Figures saved to:', OUTPUT_DIR.resolve())